# MeRIP-seq / m6A-seq related Metadata validated and modified process

- lncRNA がメチル化されているのかどうか検証するために、公共データベースのデータを活用しようと考えている
- [./12_8_metadata_other_seq.ipynb](./12_8_metadata_other_seq.ipynb)から処理を移動

In [1]:
import sys
import polars as pl
import datetime
import pathlib
from typing import Any
sys.path.append(str(pathlib.Path('../../').resolve()))
from scripts.metadata_pairing import create_paired_metadata
from IPython.display import display

pl.Config.set_fmt_str_lengths(200)
pl.Config.set_tbl_rows(4)

polars.config.Config

In [2]:
VERSION = datetime.datetime.now().strftime("%y%m")
execute_date = datetime.datetime.now().strftime("%y%m%d")
print(f"Execute Date: {execute_date}")

# create directory
pathlib.Path(f"../../Data/metadata/metadata_{VERSION}").mkdir(parents=True, exist_ok=True)

Execute Date: 261001


In [3]:
# drop columns list for iseq metadata
drop_columns_list = [
    "sra_aspera",
    "fastq_galaxy",
    "sra_galaxy",
    "fastq_ftp",
    "sra_ftp",
    "fastq_aspera"
]

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

# 1. GSE130892 MeRIP-seq (SINGLE)

- ✅ DOWNLOAD (yone-ssd)
- ✅ TRIMMING (yone-ssd)
- ✅ MAPPING (yone-ssd)
- ✅ PEAK CALL AND ANALYSIS (M3 MacBook Pro; using exomePeak2)

&nbsp;

- validate date: 2025-05-03
- validate date: 2026-03-13

&nbsp;

- [Limits in the detection of m6A changes using MeRIP/m6A-seq](https://doi.org/10.1038/s41598-020-63355-3)
- 以前の研究で使用していたRNA-seqとMeRIP-seq (methylated RNA immunoprecipitation sequencing) のデータ
- [./12_2_metadata_modified_true_no_control.ipynb](./12_2_metadata_modified_true_no_control.ipynb)から処理を移動
- 当初、[/12_8_metadata_other_seq.ipynb](./12_8_metadata_other_seq.ipynb)に記載していたが、こちらに移動

In [5]:
GSE130892_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE130892/GSE130892_PEP/GSE130892_PEP_raw.csv",
    separator=','
).with_columns(
    (pl.col("biosample").str.split("/").list.get(-1).alias("sample_accession")),
    (pl.lit("TruSeq mRNA Library Prep Kit").alias("rna_library")), # add rna_library column (Illumina) ?
    (pl.lit("https://doi.org/10.1038/s41598-020-63355-3").alias("reference_doi")) # add DOI of the reference paper column
).rename(
    {
        "sample_series_id": "gse_id",
        "big_key": "experiment_accession"
    }
).drop(
    "sra",
    "biosample"
)

# # MeRIP-seq
# GSE130892_PEP_merip_seq = GSE130892_PEP.filter(
#     pl.col("library_strategy") == "MeRIP-seq"
# )

# metadata from iseq
GSE130892_iseq = pl.read_csv(
    "../../Data/metadata/metadata_iseq/GSE130892_metadata.tsv",
    separator='\t'
).drop(
    drop_columns_list
)

GSE130892_iseq_filtered = GSE130892_iseq[[s.name for s in GSE130892_iseq if not (s.null_count() == GSE130892_iseq.height)]]

# pl.Config.set_tbl_rows(6)
# display(GSE130892_PEP_merip_seq)

In [6]:
GSE130892_combined_merip_seq = GSE130892_PEP.join(
    GSE130892_iseq_filtered,
    on=["experiment_accession", "sample_accession"],
    how="left",
    coalesce=True
)

GSE130892_combined_merip_seq.write_csv(
    "../../Data/metadata/metadata_modified/GSE130892_merip_seq_modified_metadata.tsv",
    separator='\t'
)

pl.Config.set_tbl_rows(12)
display(GSE130892_combined_merip_seq)

sample_name,protocol,organism,read_type,data_source,srr,srx,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_extract_protocol_ch1,sample_description,sample_platform_id,sample_contact_name,sample_contact_department,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,gse_id,sample_data_row_count,experiment_accession,…,run_accession,experiment_title,project_name,submission_accession,library_selection,last_updated,sra_bytes,fastq_bytes,instrument_platform,ncbi_reporting_standard,sample_description_right,fastq_md5,secondary_study_accession,read_count,study_title,sample_title_right,instrument_model,description,sra_md5,cell_type_right,base_count,library_strategy_right,library_source,library_layout,status,sample_alias,run_alias,library_construction_protocol,experiment_alias,center_name,cell_line_right,first_created,study_accession,accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,i64,str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,i64,str,…,str,str,str,str,str,str,i64,i64,str,str,str,str,str,i64,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""heatshock_total_input_1""","""cDNA""","""Homo sapiens""","""SINGLE""","""SRA""","""SRR9029562""","""SRX5806987""","""heatshock_total_input_1""","""GSM3755848""","""Public on May 09 2019""","""May 08 2019""","""May 09 2019""","""SRA""",1,"""OCI-Ly1 diffuse B-cell lymphoma cells""","""Homo sapiens""",9606,"""total RNA""","""TRIzol, Illumina, 1 x 42""","""heatshock_total_1.macs2_peaks.narrowPeak""","""GPL16791""","""A,,McIntyre""","""Molecular Life Sciences""","""University of Zurich""","""Winterthurerstrasse 190""","""Zurich""",8057,"""Switzerland""","""Illumina HiSeq 2500""","""cDNA""","""transcriptomic""","""RNA-Seq""","""GSM3755848""","""NONE""","""GSE130892""",0,"""SRX5806987""",…,"""SRR9029562""","""Illumina HiSeq 2500 sequencing: GSM3755848: heatshock_total_input_1 Homo sapiens RNA-Seq""","""MeRIP-seq for heat shock in B-cell lymphoma cells""","""SRA884323""","""cDNA""","""2019-05-10""",1486980708,2155905773,"""ILLUMINA""","""Generic""","""heatshock_total_input_1""","""effc44eb0669eb90530e84c9891f1d7d""","""SRP197091""",49614077,"""MeRIP-seq for heat shock in B-cell lymphoma cells""","""heatshock_total_input_1""","""Illumina HiSeq 2500""","""Illumina HiSeq 2500 sequencing: GSM3755848: heatshock_total_input_1 Homo sapiens RNA-Seq""","""088ee5112cc6f3b8beae3f09c3ce87e8""","""diffuse B-cell lymphoma cells""",2083791234,"""RNA-Seq""","""TRANSCRIPTOMIC""","""SINGLE""","""public""","""GSM3755848""","""GSM3755848_r1""","""TRIzol Illumina, 1 x 42""","""GSM3755848""","""GEO""","""OCI-Ly1""","""2019-05-10""","""PRJNA541915""","""SRR9029562""","""SRS4736449""","""2019-05-10""","""GSE130892"""
"""heatshock_total_ip_1""","""other""","""Homo sapiens""","""SINGLE""","""SRA""","""SRR9029563""","""SRX5806988""","""heatshock_total_IP_1""","""GSM3755849""","""Public on May 09 2019""","""May 08 2019""","""May 09 2019""","""SRA""",1,"""OCI-Ly1 diffuse B-cell lymphoma cells""","""Homo sapiens""",9606,"""total RNA""","""TRIzol, Illumina, 1 x 42""","""heatshock_total_1.macs2_peaks.narrowPeak""","""GPL16791""","""A,,McIntyre""","""Molecular Life Sciences""","""University of Zurich""","""Winterthurerstrasse 190""","""Zurich""",8057,"""Switzerland""","""Illumina HiSeq 2500""","""other""","""transcriptomic""","""OTHER""","""GSM3755849""","""NONE""","""GSE130892""",0,"""SRX5806988""",…,"""SRR9029563""","""Illumina HiSeq 2500 sequencing: GSM3755849: heatshock_total_IP_1 Homo sapiens OTHER""","""MeRIP-seq for heat shock

In [7]:
# MeRIP-seq
# note: IP vs input (RNA-seq data as background)
defined_pairs_gse130892_merip_seq = [
    ("SRR9029563", "SRR9029562"), # heatshock_total_IP_1 vs heatshock_total_input_1
    ("SRR9029565", "SRR9029564"), # heatshock_total_IP_2 vs heatshock_total_input_2
    ("SRR9029567", "SRR9029566"), # heatshock_total_IP_3 vs heatshock_total_input_3
    ("SRR9029569", "SRR9029568"), # untreated_total_IP_1 vs untreated_total_input_1
    ("SRR9029571", "SRR9029570"), # untreated_total_IP_2 vs untreated_total_input_2
    ("SRR9029573", "SRR9029572")  # untreated_total_IP_3 vs untreated_total_input_3
]

paired_metadata_df_gse130892_merip_seq = create_paired_metadata(
    combined_df=GSE130892_combined_merip_seq,
    defined_sample_pairs=defined_pairs_gse130892_merip_seq,
    sample_id_column_name="accession"
)

if paired_metadata_df_gse130892_merip_seq is not None:
    paired_metadata_df_gse130892_merip_seq.write_csv(
        "../../Data/metadata/metadata_paired/GSE130892_merip_seq_paired_metadata.tsv",
        separator='\t'
    )
    display(paired_metadata_df_gse130892_merip_seq)
else:
    # Simple message helps you notice when no pairs were produced
    print("No paired metadata to write (create_paired_metadata returned None).")

sample_heat_id,sample_control_id,sample_name,protocol,organism,read_type,data_source,srr,srx,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_extract_protocol_ch1,sample_description,sample_platform_id,sample_contact_name,sample_contact_department,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,gse_id,…,reference_doi,run_accession,experiment_title,project_name,submission_accession,library_selection,last_updated,sra_bytes,fastq_bytes,instrument_platform,ncbi_reporting_standard,sample_description_right,fastq_md5,secondary_study_accession,read_count,study_title,sample_title_right,instrument_model,description,sra_md5,cell_type_right,base_count,library_strategy_right,library_source,library_layout,status,sample_alias,run_alias,library_construction_protocol,experiment_alias,center_name,cell_line_right,first_created,study_accession,secondary_sample_accession,first_public,study_alias
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,…,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""SRR9029563""","""SRR9029562""","""heatshock_total_ip_1 vs heatshock_total_input_1""","""other vs cDNA""","""Homo sapiens""","""SINGLE""","""SRA""","""SRR9029563 vs SRR9029562""","""SRX5806988 vs SRX5806987""","""heatshock_total_IP_1 vs heatshock_total_input_1""","""GSM3755849 vs GSM3755848""","""Public on May 09 2019""","""May 08 2019""","""May 09 2019""","""SRA""","""1""","""OCI-Ly1 diffuse B-cell lymphoma cells""","""Homo sapiens""","""9606""","""total RNA""","""TRIzol, Illumina, 1 x 42""","""heatshock_total_1.macs2_peaks.narrowPeak""","""GPL16791""","""A,,McIntyre""","""Molecular Life Sciences""","""University of Zurich""","""Winterthurerstrasse 190""","""Zurich""","""8057""","""Switzerland""","""Illumina HiSeq 2500""","""other vs cDNA""","""transcriptomic""","""OTHER vs RNA-Seq""","""GSM3755849 vs GSM3755848""","""NONE""","""GSE130892""",…,"""https://doi.org/10.1038/s41598-020-63355-3""","""SRR9029563 vs SRR9029562""","""Illumina HiSeq 2500 sequencing: GSM3755849: heatshock_total_IP_1 Homo sapiens OTHER vs Illumina HiSeq 2500 sequencing: GSM3755848: heatshock_total_input_1 Homo sapiens RNA-Seq""","""MeRIP-seq for heat shock in B-cell lymphoma cells""","""SRA884323""","""other vs cDNA""","""2019-05-10""","""702554710 vs 1486980708""","""1016974814 vs 2155905773""","""ILLUMINA""","""Generic""","""heatshock_total_IP_1 vs heatshock_total_input_1""","""842b9ce07190833a3efbd744195fd408 vs effc44eb0669eb90530e84c9891f1d7d""","""SRP197091""","""23056731 vs 49614077""","""MeRIP-seq for heat shock in B-cell lymphoma cells""","""heatshock_total_IP_1 vs heatshock_total_input_1""","""Illumina HiSeq 2500""","""Illumina HiSeq 2500 sequencing: GSM3755849: heatshock_total_IP_1 Homo sapiens OTHER vs Illumina HiSeq 2500 sequencing: GSM3755848: heatshock_total_input_1 Homo sapiens RNA-Seq""","""6f3ef32435161519d0b4d1ed3b07caec vs 088ee5112cc6f3b8beae3f09c3ce87e8""","""diffuse B-cell lymphoma cells""","""968382702 vs 2083791234""","""OTHER vs RNA-Seq""","""TRANSCRIPTOMIC""","""SINGLE""","""public""","""GSM3755849 vs GSM3755848""","""GSM3755849_r1 vs GSM3755848_r1""","""TRIzol Illumina, 1 x 42""","""GSM3755849 vs GSM3755848""","""GEO""","""OCI-Ly1""","""2019-05-10""","""PRJNA541915""","""SRS4736452 vs SRS4736449""","""2019-05-10""","""GSE130892"""
"""SRR9029565""","""SRR9029564""","""heatshock_total_ip_2 vs heatshock_total_input_2""","""other vs cDNA""","""Homo sapiens""","""SINGLE""","""SRA""","""SRR9029565 vs SRR90

&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

# 2. GSE164200 (SINGLE) m6A-RIP-seq

- ✅ DOWNLOAD (yone-ssd)
- ✅ TRIMMING (yone-ssd)
- ✅ MAPPING (yone-ssd)
- ✅ PEAK CALL AND ANALYSIS (M3 MacBook Pro; using exomePeak2)

&nbsp;

- validate date: 2026-02-22

&nbsp;

- [m6 A modification of HSATIII lncRNAs regulates temperature-dependent splicing](https://doi.org/10.15252/embj.2021107976)
- [./12_7_metadata_proseq.ipynb](./12_7_metadata_proseq.ipynb)から処理を移動
- [./12_8_metadata_other_seq.ipynb](./12_8_metadata_other_seq.ipynb)から再度こちらに処理を移動

In [8]:
GSE164200_PEP = pl.read_csv(
    "../../Data/metadata/metadata_geofetch/GSE164200/GSE164200_PEP/GSE164200_PEP_raw.csv",
    separator=','
)

pl.Config.set_tbl_rows(20)
display(GSE164200_PEP)

sample_name,protocol,organism,read_type,data_source,srr,srx,sample_title,sample_geo_accession,sample_status,sample_submission_date,sample_last_update_date,sample_type,sample_channel_count,sample_source_name_ch1,sample_organism_ch1,sample_taxid_ch1,sample_molecule_ch1,sample_description,sample_platform_id,sample_contact_name,sample_contact_email,sample_contact_laboratory,sample_contact_department,sample_contact_institute,sample_contact_address,sample_contact_city,sample_contact_state,sample_contact_zip_postal_code,sample_contact_country,sample_instrument_model,sample_library_selection,sample_library_source,sample_library_strategy,gsm_id,sample_supplementary_file_1,sample_series_id,sample_data_row_count,big_key,cell_line,rna_population,treatment,genotype_variation,genome_build,supplementary_files_format_and_content,biosample,sra
str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,i64,str,str,str,str,str,str,str,str,i64,str,str,i64,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str
"""a1_37-input_rep1""","""other""","""Homo sapiens""","""SINGLE""","""SRA""","""SRR13348046""","""SRX9774087""","""A1_37-input rep1""","""GSM5002156""","""Public on Apr 30 2021""","""Jan 04 2021""","""Apr 30 2021""","""Input sample""",1,"""HeLa cell""","""Homo sapiens""",9606,"""polyA RNA""","""A1-Input""","""GPL20301""","""KENSUKE,,NINOMIYA""","""k-ninomiya@fbs.osaka-u.ac.jp""","""RNA biofunction""","""Graduate school of Frontier Biosciences""","""Osaka university""",60,"""吹田市山田丘１－３""","""大阪府""",871,"""Japan""","""Illumina HiSeq 4000""","""other""","""transcriptomic""","""RIP-Seq""","""GSM5002156""","""NONE""","""GSE164200""",0,"""SRX9774087""","""HeLa""","""poly(A) RNA""","""37 C""","""Control""","""human hg19""","""An exel file including m6A peaks for each sample""","""https://www.ncbi.nlm.nih.gov/biosample/SAMN17208964""","""https://www.ncbi.nlm.nih.gov/sra?term=SRX9774087"""
"""a1_37-ip-rep1""","""other""","""Homo sapiens""","""SINGLE""","""SRA""","""SRR13348047""","""SRX9774088""","""A1_37-IP-rep1""","""GSM5002157""","""Public on Apr 30 2021""","""Jan 04 2021""","""Apr 30 2021""","""IP sample""",1,"""HeLa cell""","""Homo sapiens""",9606,"""polyA RNA""","""A1-IP""","""GPL20301""","""KENSUKE,,NINOMIYA""","""k-ninomiya@fbs.osaka-u.ac.jp""","""RNA biofunction""","""Graduate school of Frontier Biosciences""","""Osaka university""",60,"""吹田市山田丘１－３""","""大阪府""",871,"""Japan""","""Illumina HiSeq 4000""","""other""","""transcriptomic""","""RIP-Seq""","""GSM5002157""","""NONE""","""GSE164200""",0,"""SRX9774088""","""HeLa""","""poly(A) RNA""","""37 C""","""Control""","""human hg19""","""An exel file including m6A peaks for each sample""","""https://www.ncbi.nlm.nih.gov/biosample/SAMN17208963""","""https://www.ncbi.nlm.nih.gov/sra?term=SRX9774088"""
"""a2_37-input_rep2""","""other""","""Homo sapiens""","""SINGLE""","""SRA""","""SRR13348048""","""SRX9774089""","""A2_37-input rep2""","""GSM5002158""","""Public on Apr 30 2021""","""Jan 04 2021""","""Apr 30 2021""","""Input sample""",1,"""HeLa cell""","""Homo sapiens""",9606,"""polyA RNA""","""A2-Input""","""GPL20301""","""KENSUKE,,NINOMIYA""","""k-ninomiya@fbs.osaka-u.ac.jp""","""RNA biofunction""","""Graduate school of Frontier Biosciences""","""Osaka university""",60,"""吹田市山田丘１－３""","""大阪府""",871,"""Japan""","""Illumina HiSeq 4000""","""other""","""transcriptomic""","""RIP-Seq""","""GSM5002158""","""NONE""","""GSE164200""",0,"""SRX9774089""","""HeLa""","""poly(A) RNA""","""37 C""","""Control""","""human hg19""","""An exel file including m6A peaks for each sample""","""https://www.ncbi.nlm.nih.gov/biosample/SAMN17208962""","""https://www.ncbi.nlm.nih.gov/sra?term=SRX9774089"""
"""a2_37-ip-rep2""","""other""","""Homo sapiens""","""SINGLE""","""SRA""","""SRR13348049""","""SRX9774090""","""A2_37-IP-rep2""","""GSM5002159""","""Public on Apr 30 2021""","""Jan 04 2021""","""Apr 30 2021""","""IP sample""",1,"""HeLa cell""","""Homo sapiens""",9606,"""polyA RNA""","""A2-IP""","""GPL2030